In [1]:
import os, sys
from PIL import Image
from tqdm import tqdm
import yaml


# Add the project root directory to sys.path
project_root = os.path.abspath('../..')  # Adjust this path if needed
sys.path.append(project_root)

from utils import get_images_from_list, create_file_names_list
from utils import mask_on_image, dilate_mask, plot_grid

from inpainting_flux import inpaint_flux, inpaint_batch_flux, setup_diffusion_model_flux

2025-01-12 16:21:06.832679: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-12 16:21:06.846043: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-12 16:21:06.850117: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-12 16:21:06.861873: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-12 16:21:08.521148: W tensorflow/compiler/tf2

[01/12/25 16:21:14] WARNING  Your inference package version 0.12.1 is out of date! Please upgrade to ]8;id=28969;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=356328;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py#35\35]8;;\
                             version 0.33.0 of inference for the latest features and bug fixes by                  
                             running `pip install --upgrade inference`.                                            

## Datasets
### Load ISIC 2019 dataset

In [2]:
# Load the YAML file into a dictionary
dataset_yaml_file = "/workspace/home/data/datasets.yaml"
with open(dataset_yaml_file, "r") as file:
    datasets = yaml.safe_load(file)


# get dataset
dataset_key = "ISIC2019"
dataset = datasets[dataset_key]

## Setup images, custom vocabulary and models

In [3]:
def convert_img_id(img_id):
    # 55292 -> ISIC_0055292.jpg
    # 10 -> ISIC_0000010.jpg
    return f"ISIC_{str(img_id).zfill(7)}.jpg"
mask_ids = [604, 712, 592, 610, 932, 9040, 600]

In [4]:
pipe, generator = setup_diffusion_model_flux(device = 0)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [ ]:
n = 20
segmentation_id = 'gt'
images_path = dataset["raw_path"]
#segmentation_masks_path = dataset['gt_seg_path']
segmentation_masks_path = f"/workspace/home/pipeline_outputs/{dataset_key}/{segmentation_id}/seg"
mask_names = create_file_names_list(segmentation_masks_path)[:n]
#mask_names = [convert_img_id(mask_id) for mask_id in mask_ids]
images = get_images_from_list(mask_names, images_path)
segmentation_masks = get_images_from_list(mask_names, segmentation_masks_path)

prompt = "skin tissue with a lesion"
#negative_prompt = "colorful patch"
# "remove the colorful patch and replace with skin tissue"



## Run model on segmentations and visualize the inpainting

In [ ]:
def inpaint_flux(
    mask_dict,
    raw_images_path,
    prompt,
    pipeline,
    generator,
    kernel_size=5,
    iterations=10,
    size=(299, 299),
):
    # Save the dilated mask and images to the temporary file
    mask_path = mask_dict["file_path"]
    file_name = mask_dict["file_name"]

    img = Image.open(f"{raw_images_path}/{file_name}").convert("RGB")

    mask = (
        Image.open(mask_path)
        .convert("L")
        .resize(img.size)
        .point(lambda x: 255 if x > 128 else 0)
    )
    dilated_mask = dilate_mask(
        mask,
        kernel_size=kernel_size,
        iterations=iterations,
    )
    dilated_mask = dilated_mask.resize(img.size)

    #!iopaint run --model="diffusers/stable-diffusion-xl-1.0-inpainting-0.1" --device={device} --image={img_url} --mask={mask_path} --output={output_dir_confounder2}

    image = img.resize((1024, 1024))
    mask = dilated_mask.resize((1024, 1024))
    # Run the diffuser
    result2 = pipeline(
        prompt=prompt,
        image=image,
        mask_image=mask,
        height=1632,
        width=1232,
        guidance_scale=30,
        num_inference_steps=100,
        # max_sequence_length=512,
    ).images[0]

    result2 = result2.resize(img.size)

    return result2

In [ ]:
kernel_size = 5
iterations = 3
output_dir = f"/workspace/home/pipeline_outputs/ISIC2019"
save_dir = f"{output_dir}/thesis_images"

for i, mask_dict in enumerate(tqdm(segmentation_masks)):
    # result 1: inpaint with iopaint
    # result 2: inpaint diffusion model with result 1 as input
    
    result2 = inpaint_flux(
        mask_dict,
        dataset["raw_path"],
        prompt,
        pipe,
        generator,
        kernel_size=kernel_size,
        iterations=iterations,
    )

    # visualize output
    print(f"Image: {mask_dict['file_name']}")
    img = Image.open(images[i]['file_path']) 
    mask = Image.open(mask_dict['file_path']).convert("L").point(lambda x: 255 if x > 128 else 0)
    dilated_mask = dilate_mask(mask, kernel_size=kernel_size, iterations=iterations)
    # Resize the mask to the size of the image
    mask = mask.resize(img.size)
    dilated_mask = dilated_mask.resize(img.size)
    mask_on_img = mask_on_image(img, mask)
    dilated_mask_on_img = mask_on_image(img, dilated_mask)
    plot_grid([img, mask_on_img, dilated_mask_on_img, result2], label=["Original", "Mask", "Dilated Mask", "Flux"], ncols=3)
    #result2.resize((400,400)).save(f"{save_dir}/flux_{mask_dict['file_name'].replace('.jpg', '.png')}")
    #dilated_mask_on_img.resize((400,400)).save(f"{save_dir}/dilated_mask_{segmentation_id}_{mask_dict['file_name']}")

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]